# 1D J2J2 inference:  PoincareRNN (seed 111-555) 

This is part of the work arxiv:2604.24337 [quant-ph] by HL Dao. For the purpose of reproducing the results, the paths to the trained weights have to be adjusted to match the repo structure. 

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.append('../../1_hypnqs_torch_poincare/utility_poincare')
from j1j2_hyprnn_train_loop import *
numsamples = 10000

GPU Available:  False


In [2]:
def clip_local_energies(eloc, threshold=5.0):
    # Convert to numpy if it's a torch tensor, or vice versa
    eloc_real = np.real(eloc)
    median = np.median(eloc_real)
    mad = np.median(np.abs(eloc_real - median))
    
    # Standard safety check to avoid division by zero if MAD is 0
    if mad == 0:
        return eloc
        
    lower_bound = median - threshold * mad
    upper_bound = median + threshold * mad
    
    # Clip the values (keeping the imaginary part if it exists)
    # We create a copy to avoid modifying the original array in place
    clipped = np.clip(eloc_real, lower_bound, upper_bound)
    
    # If the original was complex, restore the imaginary part
    if np.iscomplexobj(eloc):
        return clipped + 1j * np.imag(eloc)
    return clipped  

def define_load_test(wf,  numsamples,path_to_weights, Ee, clipped_e=False):
    state_dict = torch.load(path_to_weights, map_location=torch.device('cpu'))   
    new_state_dict = {}
    for key, value in state_dict.items():
        # Strip prefixes and rename keys to match current architecture
        new_key = key.replace('model.', '').replace('cell.', 'rnn.')
        new_state_dict[new_key] = value
    # This line loads the RE-MAPPED weights
    wf.model.load_state_dict(new_state_dict, strict=False)
    print("Successfully remapped and loaded weights.")

    with torch.no_grad():
        test_samples_after = wf.sample(numsamples)
        if clipped_e:
            # 1. Get raw energies
            raw_gs_after = J1J2_local_energies(wf, N, J1, J2, Bz, numsamples, test_samples_after, Marshall_sign=True)
    
            # 2. APPLY CLIPPING
            test_gs_after = clip_local_energies(raw_gs_after, threshold=5.0)
    
            # 3. Calculate statistics on cleaned data
            gs_mean_a = np.round(np.mean(test_gs_after), 4)
            gs_var_a = np.round(np.var(test_gs_after), 4)
    
            # Optional: Count how many were clipped to see if the model is unstable
            num_clipped = np.sum(np.real(raw_gs_after) != np.real(test_gs_after))
            print(f"Clipped {num_clipped} outlier samples out of {numsamples}")
        else:
            test_gs_after =  J1J2_local_energies(wf, N, J1, J2, Bz, numsamples, test_samples_after, Marshall_sign=True)
            gs_mean_a = np.round(np.mean(test_gs_after),4)
            gs_var_a = np.round(np.var(test_gs_after),4)
    
    
    #wf.model.summary()
    #print('====================================================================')
    print(f'After loading weights, the ground state energy mean and variance are:')
    print(f'Mean E = {gs_mean_a}, var E = {gs_var_a}')
    print(f'DMRG energy  is {np.round(Ee,4)}')

In [3]:
N=100
units = 70
syssize = 100
J1_ = 1.0
J1 = +J1_ * np.ones(syssize)
Bz = +0.0 * np.ones(syssize)
fname=f'results_PoincareRNN'

## J2=0.0

In [4]:
J2_ = 0.0
J2 = +J2_ * np.ones(syssize)
Ee_00=-44.12773986967251
rmax=0.618

In [5]:
seed=111
hRNN_00 = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units,r_max=rmax, seed=seed)
total_params = sum(p.numel() for p in hRNN_00.model.parameters())
print(f'Total params = {total_params}')
h_wl = f'{fname}/J2={J2_}/seed={seed}/N100_J1=1.0|J2={J2_}_HypRNN_70_id_hyp_rmax=0.618_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(hRNN_00, numsamples, h_wl, Ee_00)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 5394
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-42.591400146484375+0.000699999975040555j), var E = 1.9800000190734863
DMRG energy  is -44.1277
Time taken=0.103 hrs


In [8]:
seed=222
hRNN_00 = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units,r_max=rmax, seed=seed)
total_params = sum(p.numel() for p in hRNN_00.model.parameters())
print(f'Total params = {total_params}')
h_wl = f'{fname}/J2={J2_}_rmax={rmax}/seed={seed}/N100_J1=1.0|J2={J2_}_HypRNN_70_id_hyp_rmax=0.618_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(hRNN_00, numsamples, h_wl, Ee_00, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 5394
Successfully remapped and loaded weights.
Clipped 77 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-42.133201599121094+0.01209999993443489j), var E = 3.135200023651123
DMRG energy  is -44.1277
Time taken=0.051 hrs


In [9]:
seed=333
hRNN_00 = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units,r_max=rmax, seed=seed)
total_params = sum(p.numel() for p in hRNN_00.model.parameters())
print(f'Total params = {total_params}')
h_wl = f'{fname}/J2={J2_}_rmax={rmax}/seed={seed}/N100_J1=1.0|J2={J2_}_HypRNN_70_id_hyp_rmax=0.618_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(hRNN_00, numsamples, h_wl, Ee_00, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 5394
Successfully remapped and loaded weights.
Clipped 81 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-40.729000091552734+0.0010000000474974513j), var E = 5.704500198364258
DMRG energy  is -44.1277
Time taken=0.049 hrs


In [8]:
seed=444
hRNN_00 = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units,r_max=rmax, seed=seed)
total_params = sum(p.numel() for p in hRNN_00.model.parameters())
print(f'Total params = {total_params}')
h_wl = f'{fname}/J2={J2_}/seed={seed}/N100_J1=1.0|J2=0.0_HypRNN_70_id_hyp_rmax=0.618_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(hRNN_00, numsamples, h_wl, Ee_00)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 5394
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-42.08359909057617+0.004600000102072954j), var E = 3.2939000129699707
DMRG energy  is -44.1277
Time taken=0.103 hrs


In [9]:
seed=555
hRNN_00 = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units,r_max=rmax, seed=seed)
total_params = sum(p.numel() for p in hRNN_00.model.parameters())
print(f'Total params = {total_params}')
h_wl = f'{fname}/J2={J2_}/seed={seed}/N100_J1=1.0|J2=0.0_HypRNN_70_id_hyp_rmax=0.618_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(hRNN_00, numsamples, h_wl, Ee_00)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 5394
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-42.11180114746094-9.999999747378752e-05j), var E = 3.421299934387207
DMRG energy  is -44.1277
Time taken=0.105 hrs


## J2=0.2

In [12]:
J2_ = 0.2
J2 = +J2_ * np.ones(syssize)
Ee_02=-40.7388
rmax=0.618

In [13]:
seed=111
hRNN_00 = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units,r_max=rmax, seed=seed)
total_params = sum(p.numel() for p in hRNN_00.model.parameters())
print(f'Total params = {total_params}')
h_wl = f'{fname}/J2={J2_}/seed={seed}/N100_J1=1.0|J2={J2_}_HypRNN_70_id_hyp_rmax=0.618_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(hRNN_00, numsamples, h_wl, Ee_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 5394
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-39.60200119018555-0.0010999999940395355j), var E = 0.9998999834060669
DMRG energy  is -40.7388
Time taken=0.114 hrs


In [11]:
seed=222
hRNN_00 = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units,r_max=rmax, seed=seed)
total_params = sum(p.numel() for p in hRNN_00.model.parameters())
print(f'Total params = {total_params}')
h_wl = f'{fname}/J2={J2_}/seed={seed}/N100_J1=1.0|J2={J2_}_HypRNN_70_id_hyp_rmax=0.618_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(hRNN_00, numsamples, h_wl, Ee_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 5394
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-39.592098236083984-0.00019999999494757503j), var E = 1.274899959564209
DMRG energy  is -40.7388
Time taken=0.142 hrs


In [16]:
seed=333
hRNN_00 = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units,r_max=rmax, seed=seed)
total_params = sum(p.numel() for p in hRNN_00.model.parameters())
print(f'Total params = {total_params}')
h_wl = f'{fname}/J2={J2_}_rmax={rmax}/seed={seed}/N100_J1=1.0|J2={J2_}_HypRNN_70_id_hyp_rmax=0.618_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(hRNN_00, numsamples, h_wl, Ee_02, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 5394
Successfully remapped and loaded weights.
Clipped 72 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-38.14500045776367-0.0026000000070780516j), var E = 4.184899806976318
DMRG energy  is -40.7388
Time taken=0.071 hrs


In [13]:
seed=444
hRNN_00 = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units,r_max=rmax, seed=seed)
total_params = sum(p.numel() for p in hRNN_00.model.parameters())
print(f'Total params = {total_params}')
h_wl = f'{fname}/J2={J2_}/seed={seed}/N100_J1=1.0|J2={J2_}_HypRNN_70_id_hyp_rmax=0.618_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(hRNN_00, numsamples, h_wl, Ee_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 5394
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-39.583499908447266-0.004100000020116568j), var E = 1.3569999933242798
DMRG energy  is -40.7388
Time taken=0.139 hrs


In [14]:
seed=555
hRNN_00 = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units,r_max=rmax, seed=seed)
total_params = sum(p.numel() for p in hRNN_00.model.parameters())
print(f'Total params = {total_params}')
h_wl = f'{fname}/J2={J2_}/seed={seed}/N100_J1=1.0|J2={J2_}_HypRNN_70_id_hyp_rmax=0.618_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(hRNN_00, numsamples, h_wl, Ee_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 5394
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-39.21099853515625+0.002400000113993883j), var E = 1.9098999500274658
DMRG energy  is -40.7388
Time taken=0.141 hrs


## J2=0.5, rmax=0.8

Only 222 and 555 converged. 

In [31]:
J2_ = 0.5
J2 = +J2_ * np.ones(syssize)
Ee_05=37.5000
rmax=0.8

In [16]:
seed=222
hRNN_00 = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units,r_max=rmax, seed=seed)
total_params = sum(p.numel() for p in hRNN_00.model.parameters())
print(f'Total params = {total_params}')
h_wl = f'{fname}/J2={J2_}_rmax={rmax}/seed={seed}/N100_J1=1.0|J2={J2_}_HypRNN_70_id_hyp_rmax={rmax}_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(hRNN_00, numsamples, h_wl, Ee_05, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 5394
Successfully remapped and loaded weights.
Clipped 226 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-35.88140106201172+9.999999747378752e-05j), var E = 0.694599986076355
DMRG energy  is 37.5
Time taken=0.202 hrs


In [18]:
# Seed 555: Best model saved at epoch 829 with best E=-36.31379-0.15427j, varE=14.08536
seed=555
hRNN_00 = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units,r_max=rmax, seed=seed)
total_params = sum(p.numel() for p in hRNN_00.model.parameters())
print(f'Total params = {total_params}')
h_wl = f'{fname}/J2={J2_}_rmax={rmax}/seed={seed}/N100_J1=1.0|J2={J2_}_HypRNN_70_id_hyp_rmax={rmax}_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(hRNN_00, numsamples, h_wl, Ee_05, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 5394
Successfully remapped and loaded weights.
Clipped 375 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-35.92530059814453-0.0010000000474974513j), var E = 0.6155999898910522
DMRG energy  is 37.5
Time taken=0.201 hrs


## J2=0.8, rmax=0.95

lr2=2e-3, seed 222, 444 didnt converge.

In [25]:
J2_ = 0.8
J2 = +J2_ * np.ones(syssize)
Ee_08=-42.07006297371643
rmax=0.95

In [28]:
seed=111
rmax=0.95
hRNN_00 = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units,r_max=rmax, seed=seed)
total_params = sum(p.numel() for p in hRNN_00.model.parameters())
print(f'Total params = {total_params}')
h_wl = f'{fname}/J2={J2_}_rmax={rmax}_lr2=5e-3/seed={seed}/N100_J1=1.0|J2={J2_}_HypRNN_70_id_hyp_rmax={rmax}_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(hRNN_00, numsamples, h_wl, Ee_08, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 5394
Successfully remapped and loaded weights.
Clipped 92 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-33.645999908447266-0.006200000178068876j), var E = 11.824999809265137
DMRG energy  is -42.0701
Time taken=0.13 hrs


In [22]:
seed=333
rmax=0.95
hRNN_00 = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units,r_max=rmax, seed=seed)
total_params = sum(p.numel() for p in hRNN_00.model.parameters())
print(f'Total params = {total_params}')
h_wl = f'{fname}/J2={J2_}/seed={seed}/N100_J1=1.0|J2={J2_}_HypRNN_70_id_hyp_rmax={rmax}_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(hRNN_00, numsamples, h_wl, Ee_08)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 5394
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-36.76890182495117+0.013799999840557575j), var E = 8.213000297546387
DMRG energy  is -42.0701
Time taken=0.161 hrs


In [30]:
seed=555
hRNN_00 = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units,r_max=rmax, seed=seed)
total_params = sum(p.numel() for p in hRNN_00.model.parameters())
print(f'Total params = {total_params}')
h_wl = f'{fname}/J2={J2_}_rmax={rmax}_lr2=5e-3/seed={seed}/N100_J1=1.0|J2={J2_}_HypRNN_70_id_hyp_rmax={rmax}_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(hRNN_00, numsamples, h_wl, Ee_08, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 5394
Successfully remapped and loaded weights.
Clipped 231 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-38.468101501464844+0.0035000001080334187j), var E = 3.3884999752044678
DMRG energy  is -42.0701
Time taken=0.083 hrs
